# 第五轮：固定容量、分布式状态空间记忆基线
这一轮仍用离散频带任务，不涉及真实音频。每个频带片段现在随机持续 3–8 帧，句中停顿也变化。模型用多组旋转、衰减的连续状态把历史逐帧编码到固定大小的向量中；训练时加入近期输入恢复探针。探针只用于训练，推理时不向模型提供原始历史缓存。输出仍是 9 个频带／静音类别。
训练损失采用 CTC，对按顺序产生的频带片段求和，不规定它们必须持续与输入相同的帧数，也不固定开始时间；另以较轻的总发声时长约束避免把整句压成几个极短脉冲，并惩罚在输入仍发声时输出。评测使用带片段时长惩罚的单调编辑相似度：同一频带输出 7 帧比输出 1 帧更接近输入 8 帧。

达标门槛先设为：0.15 回传下总体时长容忍的有序相似度 ≥0.95，含重复样本 ≥0.90，长度 6 样本 ≥0.90，不抢话比例 ≥0.99；再检查 0 和 0.35 回传条件。门槛是本项目的工程目标，不是论文保证。未达标时继续研究离散任务，不进入真实音频阶段。

依次运行：① 选择 Colab GPU；② 从 GitHub 获取代码；③ 挂载 Drive；④ 快速检查；⑤ 正式训练；⑥ 与保存的第四轮模型在同一新测试集上比较；⑦ 打包回传 `MyDrive/Parrot/trial5_results.zip`。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
assert (repo / 'trial5.py').is_file()
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码提交:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/Parrot')
result_dir = drive_root / 'trial5_results'
result_dir.mkdir(parents=True, exist_ok=True)
print(result_dir)


先确认时长相似度的行为，再运行快速检查。快速检查只验证前向、反向和指标导出，不代表模型达标。


In [ ]:
from trial5 import duration_aware_similarity
reference = [5] * 8
scores = {n: duration_aware_similarity(reference, [5] * n) for n in (8, 7, 6, 1)}
print('输入频带 5 持续 8 帧，对应输出时长评分:', scores)
assert scores[8] > scores[7] > scores[6] > scores[1]


In [ ]:
!python trial5.py --smoke --batch 3 --out /content/Parrot/trial5_smoke


正式训练默认 3000 步。模型较前几轮更大，训练可能较久；每 100 步日志会写入 Drive。若 Colab 中断，请保留并回传已写入的 `train_log.csv`。


In [ ]:
!python trial5.py --steps 3000 --batch 20 --eval 300 --out {result_dir}


读取第五轮结果。相似度允许适度时长变化，仍惩罚频带错误、漏段、重段与极端时长差异；它不是最终目标的真实声学相似性。特别查看重复符号和长度 6–8 的结果。


In [ ]:
import json
from IPython.display import Audio, display
path = result_dir / 'metrics.json'
report = json.loads(path.read_text())
report['git_commit'] = commit
path.write_text(json.dumps(report, indent=2))
for gain, groups in report['conditions'].items():
    print('回传增益:', gain)
    for name in ('overall', 'repeated', 'distinct', 'length_6', 'length_7', 'length_8'):
        if name in groups: print(name, groups[name])
display(Audio(filename=str(result_dir / 'input.wav')))
display(Audio(filename=str(result_dir / 'model.wav')))


如第四轮权重仍在 Drive，下一格会用**同一批新测试样本**计算其时长容忍相似度，作为历史模型参考。第四轮只训练过固定两帧音节，本轮含 3–8 帧音节；两轮模型规模和训练步数也不同，因此这只是分布变化下的参考，不能把差异全归因于状态空间结构。


In [ ]:
import torch
from trial2 import StreamMemory
from trial5 import evaluate
old_weights = drive_root / 'trial4_results' / 'aligned' / 'model.pt'
if old_weights.is_file():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    old = StreamMemory().to(device)
    old.load_state_dict(torch.load(old_weights, map_location=device, weights_only=True))
    reference, _ = evaluate(old, device, 7171, 300, 0.15)
    (result_dir / 'trial4_reference.json').write_text(json.dumps(reference, indent=2))
    print('第四轮历史模型，新测试集：', reference['overall'])
else:
    print('未找到第四轮权重，跳过历史比较')


In [ ]:
import shutil
archive = shutil.make_archive(str(drive_root / 'trial5_results'), 'zip', root_dir=str(drive_root), base_dir='trial5_results')
print('请回传:', archive)
